# Imports

In [4]:
pip install lxml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 86.5 MB/s  0:00:00

[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
pip install tqdm


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
from lxml import etree
import pandas as pd
import os
from IPython.display import clear_output
from tqdm import tqdm
from shutil import copyfile

# Sélection des séances

Voici deux fonctions qui retiennent la liste des séances ayant un point lié aux VSS dans leur sommaire, ou 3 références au champ lexical des VSS dans leur texte.

In [5]:
def tri_titres(fichier, liste):
    """
        Pour un fichier de débats à l'assemblée, teste s'il a ou non un titre spécifique dans son sommaire.
        input : un nom de fichier .xml au format str et une liste de chaines de caractères
        output : un booléen True si le sommaire comprend une des expressions
    """
    root=etree.parse(fichier).getroot()
    yes=False
    for chaine in liste:
        for titre1 in root.findall("{http://schemas.assemblee-nationale.fr/referentiel}metadonnees/{http://schemas.assemblee-nationale.fr/referentiel}sommaire/{http://schemas.assemblee-nationale.fr/referentiel}sommaire1/{http://schemas.assemblee-nationale.fr/referentiel}titreStruct/{http://schemas.assemblee-nationale.fr/referentiel}intitule"):
            #print(titre1.text)
            if chaine in titre1.text.lower():
                yes=True
                break
        for titre2 in root.findall("{http://schemas.assemblee-nationale.fr/referentiel}metadonnees/{http://schemas.assemblee-nationale.fr/referentiel}sommaire/{http://schemas.assemblee-nationale.fr/referentiel}sommaire1/{http://schemas.assemblee-nationale.fr/referentiel}sommaire2/{http://schemas.assemblee-nationale.fr/referentiel}titreStruct/{http://schemas.assemblee-nationale.fr/referentiel}intitule"):
            #print(titre2.text)
            if titre2.text!=None and chaine in titre2.text.lower():
                yes=True
                break
    return yes
    

In [10]:
def tri_textes(fichier, liste, seuil):
    """
        Pour un fichier de débats à l'assemblée, teste s'il a un certain nombre d'expressions d'une liste renseignée dans le texte.
        input : un nom de fichier .xml au format str, une liste de chaines à tester, un nombre d'occurences à partir duquel on garde le texte
        output : un booléen True si le texte comprend n_occurrences expressions

        attention : l'apostrophe est notée "’" dans les débats
    """
    root=etree.parse(fichier).getroot()
    yes=False
    nb_occurences=0
    for chaine in liste:
        for i in root.findall("{http://schemas.assemblee-nationale.fr/referentiel}contenu/{http://schemas.assemblee-nationale.fr/referentiel}point/{http://schemas.assemblee-nationale.fr/referentiel}paragraphe") + root.findall("{http://schemas.assemblee-nationale.fr/referentiel}contenu/{http://schemas.assemblee-nationale.fr/referentiel}point/{http://schemas.assemblee-nationale.fr/referentiel}interExtraction/{http://schemas.assemblee-nationale.fr/referentiel}paragraphe"):
            if chaine in ''.join(i.findall("{http://schemas.assemblee-nationale.fr/referentiel}texte")[0].itertext()).lower():
                nb_occurences+=1
            
    if nb_occurences >= seuil:
        yes=True
    return yes

In [27]:
a_tester=["violences sexistes", "violences sexuelles", "violences conjugales", "violences faites aux femmes", "féminicide", "viol ", "viols", "harcèlement sexuel", "inceste"]
liste_pertinente=[] # liste des fichiers contenant les termes de la liste a_tester
seuil=3 # Nombre de termes de la liste à avoir pour garder un texte

In [33]:
# Recherche dans la XVe législature
if os.getcwd() !="/home/onyxia/work/projet_eco_socio/data/CompteRendusXV/":
    os.chdir("/home/onyxia/work/projet_eco_socio/data/CompteRendusXV/")
for file in tqdm(os.scandir()):
    if file.name[-3:]=="xml":
        if tri_titres(file.name, a_tester)==True:
            liste_pertinente.append(file.name)
        if tri_textes(file.name, a_tester, seuil)==True:
            liste_pertinente.append(file.name)

# Recherche dans la XVIe législature
if os.getcwd() !="/home/onyxia/work/projet_eco_socio/data/CompteRendusXVI/":
    os.chdir("/home/onyxia/work/projet_eco_socio/data/CompteRendusXVI/")
for file in tqdm(os.scandir()):
    if file.name[-3:]=="xml":
        if tri_titres(file.name, a_tester)==True:
            liste_pertinente.append(file.name)
        if tri_textes(file.name, a_tester, seuil)==True:
            liste_pertinente.append(file.name)


    # Recherche dans la XVIIe législature
if os.getcwd() !="/home/onyxia/work/projet_eco_socio/data/CompteRendusXVII/":
    os.chdir("/home/onyxia/work/projet_eco_socio/data/CompteRendusXVII/")
for file in tqdm(os.scandir()):
    if file.name[-3:]=="xml":
        if tri_titres(file.name, a_tester)==True:
            liste_pertinente.append(file.name)
        if tri_textes(file.name, a_tester, seuil)==True:
            liste_pertinente.append(file.name)

liste_pertinente=list(set(liste_pertinente)) # on enlève les doublons

1563it [00:58, 26.64it/s]
605it [00:29, 20.35it/s]
365it [00:17, 21.16it/s]


In [34]:
# Nombre de séances retenues
len(liste_pertinente)

296

## Ajout des séances retenues dans un dossier séparé

In [46]:
# XVe législature
if os.getcwd() !="/home/onyxia/work/projet_eco_socio/data/CompteRendusXV/":
    os.chdir("/home/onyxia/work/projet_eco_socio/data/CompteRendusXV/")
for file in tqdm(os.scandir()):
    if file.name in liste_pertinente:
        copyfile(file.name, "/home/onyxia/work/projet_eco_socio/sorted/xv/" + file.name)

# XVIe législature
if os.getcwd() !="/home/onyxia/work/projet_eco_socio/data/CompteRendusXVI/":
    os.chdir("/home/onyxia/work/projet_eco_socio/data/CompteRendusXVI/")
for file in tqdm(os.scandir()):
    if file.name in liste_pertinente:
        copyfile(file.name, "/home/onyxia/work/projet_eco_socio/sorted/xvi/" + file.name)

# XVe législature
if os.getcwd() !="/home/onyxia/work/projet_eco_socio/data/CompteRendusXVII/":
    os.chdir("/home/onyxia/work/projet_eco_socio/data/CompteRendusXVII/")
for file in tqdm(os.scandir()):
    if file.name in liste_pertinente:
        copyfile(file.name, "/home/onyxia/work/projet_eco_socio/sorted/xvii/" + file.name)

1563it [00:00, 18716.45it/s]
605it [00:00, 13226.07it/s]
365it [00:00, 14786.51it/s]


Objectif : pour chaque document traité, obtenir le sommaire. Cela permet de trier entre les documents qui abordent des thématiques qui nous intéressent (les violences sexistes et sexuelles) et ceux qui n'ont pas d'intérêt dans notre analyse.

# Parsing des débats à l'Assemblée Nationale

La fonction ci-dessous crée un dataframe regroupant les textes des débats, avec quelques informations complémentaires (date, législature...) et les id des députés. Il est possible qu'elle ne parse pas parfaitement tous les débats, mais on peut rajouter des chemins dans la boucle for pour compléter les manques.

In [88]:
def parsing_debats(fichier):
    """
        Parse un fichier .xml de débats à l'Assemblée Nationale et crée un dataframe des prises de paroles, avec quelques informations complémentaires.
        input : un fichier.xml
        output : un dataframe Pandas
    """
    # Lecture du fichier
    tree=etree.parse(fichier)
    root=tree.getroot()

    # Pour faciliter la recherche
    adresse = "{http://schemas.assemblee-nationale.fr/referentiel}"
    
    # Création du dataframe rassemblant les prises de paroles lors de la séance
    df=pd.DataFrame({"seanceRef":[], "ordre_absolu_seance":[], "date":[], "legislature":[], "id_acteur":[], "code_grammaire":[], "texte":[]})
    df["seanceRef"]=df["seanceRef"].astype("str")
    df["date"]=df["date"].astype("str")
    df["legislature"]=df["legislature"].astype("str")
    df["id_acteur"]=df["id_acteur"].astype("str")
    df["code_grammaire"]=df["code_grammaire"].astype("str")
    df["texte"]=df["texte"].astype("str")
    
    # remplissage du dataframe
    k=0
    for i in root.findall(adresse + "contenu/" + adresse + "point/" + adresse + "interExtraction/") + root.findall(adresse + "contenu/" + adresse + "point/" + adresse + "point/" + adresse + "interExtraction/") + root.findall(adresse + "contenu/" + adresse + "point/" + adresse + "point/" + adresse + "point/" + adresse + "point/" + adresse + "interExtraction/") + root.findall(adresse + "contenu/" + adresse + "point/" + adresse + "paragraphe"):
        df.loc[k, "seanceRef"]=root.findall(adresse + "uid")[0].text
        df.loc[k, "ordre_absolu_seance"]=i.get("ordre_absolu_seance")
        df.loc[k, "date"]=root.findall(adresse + "metadonnees/" + adresse + "dateSeance")[0].text
        df.loc[k, "legislature"]=root.findall(adresse + "metadonnees/" + adresse + "legislature")[0].text
        df.loc[k, "id_acteur"]=i.get("id_acteur")
        df.loc[k, "code_grammaire"]=i.get("code_grammaire")
        chaine=""
        for j in i.findall(adresse + "texte"):
            parts = []
        
            if j.text:
                parts.append(j.text)
        
            for child in j:
                if not child.tag.endswith("italique") and child.text:
                    parts.append(child.text)
                if child.tail:
                    parts.append(child.tail)
        
            chaine = "".join(parts)
            chaine = chaine.replace("\xa0", " ")
        df.loc[k, "texte"]=chaine
        k+=1
    df["ordre_absolu_seance"]=df["ordre_absolu_seance"].astype("int")
    df=df.sort_values(by="ordre_absolu_seance")
    return df

# Parsing des députés et partis politiques (à finir)

On sélectionne les députés des 3 législatures qui nous intéressent, et leur appartenance politique.

In [ ]:
adresse = "{http://schemas.assemblee-nationale.fr/referentiel}"

Les cellules suivantes ont été exécutées en local pour éviter d'importer des milliers de fichiers. Le résultat (fichiers des députés) est disponible dans le dossier "deputes".

In [152]:
os.chdir("/home/onyxia/work/projet_eco_socio/organes/")

for file in tqdm(os.scandir()):
    if file.name[-3:]=="xml":
        tree=etree.parse(file.name)
        root=tree.getroot()
        if root.findall(adresse + "codeType")[0].text=="PARPOL":
            copyfile(file.name, "/home/onyxia/work/projet_eco_socio/partis/" + file.name)


2085it [00:00, 18709.43it/s]


In [153]:
# DF des partis politiques
os.chdir("/home/onyxia/work/projet_eco_socio/partis/")
df_partis=pd.DataFrame({"id":[], "Nom":[]})
k=0
for file in tqdm(os.scandir()):
    if file.name[-3:]=="xml":
        tree=etree.parse(file.name)
        root=tree.getroot()
        df_partis.loc[k, "id"]=root.findall(adresse + "uid")[0].text
        df_partis.loc[k, "Nom"]=root.findall(adresse + "libelle")[0].text
        k+=1
        

0it [00:00, ?it/s]/tmp/ipykernel_4961/436329637.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'PO744856' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_partis.loc[k, "id"]=root.findall(adresse + "uid")[0].text
/tmp/ipykernel_4961/436329637.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'La France Insoumise' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_partis.loc[k, "Nom"]=root.findall(adresse + "libelle")[0].text
8it [00:00, 618.38it/s]


In [151]:
# Législatures de l'Assemblée 
os.chdir("/home/onyxia/work/projet_eco_socio/organes/")

for file in tqdm(os.scandir()):
    if file.name[-3:]=="xml":
        tree=etree.parse(file.name)
        root=tree.getroot()
        if root.findall(adresse + "codeType")[0].text=="ASSEMBLEE":
            copyfile(file.name, "/home/onyxia/work/projet_eco_socio/assemblee/" + file.name)


802it [00:00, 18984.43it/s]


In [ ]:
# Sélection des députés
os.chdir("/home/onyxia/work/projet_eco_socio/acteur")
liste_legislatures=["PO791932", "PO838901", "PO717460"]

for file in os.scandir():
    if file.name[-3:]=="xml":
        tree=etree.parse(file.name)
        root=tree.getroot()
        for i in root.findall(adresse + "mandats/" + adresse + "mandat/" + adresse + "organes/" + adresse + "organeRef"):
            if i.text in liste_legislatures:
                copyfile(file.name, "/home/onyxia/work/projet_eco_socio/deputes/" + file.name)
                break

In [28]:
os.chdir("/home/onyxia/work/projet_eco_socio/deputes")


1177
